In [4]:
%cd ..

c:\Users\Nourhan Yehia\Desktop\Jupyter\xray_pneumonia_classifier


In [5]:
import os
from pathlib import Path
import pandas as pd
from glob import glob
from PIL import Image
import shutil
from tqdm import tqdm

In [6]:
DATA_DIR = Path("data/chest_xray")

In [7]:
for split in ["train", "val", "test"]:
    print(split, ":", os.listdir(DATA_DIR / split))

train : ['NORMAL', 'PNEUMONIA']
val : ['NORMAL', 'PNEUMONIA']
test : ['NORMAL', 'PNEUMONIA']


In [8]:
def build_data_map(folder_path):
    imgs_paths = glob(str(folder_path / "*.jpeg"))
    
    data = []
    for path in imgs_paths:
        data.append({
            "img_path": path,
            "class": folder_path.name,
            "split": folder_path.parent.name
        })
    
    return pd.DataFrame(data)

In [9]:
all_dfs = []

for split_path in DATA_DIR.iterdir():
    if split_path.is_dir():
        for class_path in split_path.iterdir():
            if class_path.is_dir():
                df = build_data_map(class_path)
                all_dfs.append(df)

final_df = pd.concat(all_dfs, ignore_index=True)
final_df.head()

,img_path,class,split
0,data\chest_xray\test\NORMAL\IM-0001-0001.jpeg,NORMAL,test
1,data\chest_xray\test\NORMAL\IM-0003-0001.jpeg,NORMAL,test
2,data\chest_xray\test\NORMAL\IM-0005-0001.jpeg,NORMAL,test
3,data\chest_xray\test\NORMAL\IM-0006-0001.jpeg,NORMAL,test
4,data\chest_xray\test\NORMAL\IM-0007-0001.jpeg,NORMAL,test


In [10]:
final_df["class"].value_counts()

class
PNEUMONIA    4273
NORMAL       1583
Name: count, dtype: int64

In [11]:
BASE_DIR = Path("data/processed2")

for _, row in tqdm(final_df.iterrows(), total=len(final_df)):
    src = row["img_path"]
    split = row["split"]
    cls = row["class"]

    filename = Path(src).name

    dst_dir = BASE_DIR / split / cls
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst = dst_dir / filename

    img = Image.open(src).convert("L")
    img = img.resize((224, 224))
    img.save(dst)

print("✔ processed dataset built")

100%|██████████| 5856/5856 [01:03<00:00, 92.75it/s] 

✔ processed dataset built


In [12]:
final_df["processed_path"] = final_df.apply(
    lambda row: f"data/processed2/{row['split']}/{row['class']}/{Path(row['img_path']).name}",
    axis=1
)

In [13]:
final_df.to_csv("data/data_map2.csv", index=False)

In [14]:
sizes = []

for path in final_df["img_path"].sample(200):
    img = Image.open(path)
    sizes.append(img.size)

pd.Series(sizes).value_counts().head()

(1384, 864)     2
(1032, 680)     1
(1890, 1357)    1
(1128, 568)     1
(1426, 967)     1
Name: count, dtype: int64